In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

In [2]:
DATA_DIR = Path("../data")
DB_PATH = DATA_DIR / "crime_copilot.db"

print(DB_PATH)

..\data\crime_copilot.db


In [3]:
crimes = pd.read_csv(
    DATA_DIR / "crimes_features.csv"
)

crimes['date'] = pd.to_datetime(
    crimes['date'],
    errors='coerce'
)

print(crimes.shape)
print(crimes['date'].min())
print(crimes['date'].max())

(1388164, 27)
2021-01-01 00:00:00
2026-10-03 00:00:00


In [4]:
crime_summary = pd.DataFrame({
    'metric': [
        'total_crimes',
        'start_date',
        'end_date',
        'crime_types'
    ],

    'value': [
        len(crimes),
        crimes['date'].min().date(),
        crimes['date'].max().date(),
        crimes['primary_type'].nunique()
    ]
})

crime_summary

,metric,value
0,total_crimes,1388164
1,start_date,2021-01-01
2,end_date,2026-10-03
3,crime_types,31


In [5]:
monthly_crimes = (
    crimes
    .groupby('year_month')
    .size()
    .reset_index(
        name='crime_count'
    )
)

monthly_crimes.head()

,year_month,crime_count
0,2021-01,16347
1,2021-02,13176
2,2021-03,16237
3,2021-04,15741
4,2021-05,17912


In [6]:
crime_types = (
    crimes
    .groupby('primary_type')
    .agg(
        crime_count=('id', 'count'),
        arrests=('arrest', 'sum')
    )
    .reset_index()
)

crime_types['percentage'] = (
    crime_types['crime_count']
    / crime_types['crime_count'].sum()
    * 100
)

crime_types = (
    crime_types
    .sort_values(
        'crime_count',
        ascending=False
    )
    .reset_index(drop=True)
)

crime_types.head(10)

,primary_type,crime_count,arrests,percentage
0,THEFT,307020,18982,22.116983
1,BATTERY,247089,41129,17.799698
2,CRIMINAL DAMAGE,156492,5857,11.273308
3,ASSAULT,124873,13568,8.995551
4,MOTOR VEHICLE THEFT,114135,3495,8.222011
5,DECEPTIVE PRACTICE,94790,2547,6.828444
6,OTHER OFFENSE,90196,14782,6.497503
7,BURGLARY,50518,2408,3.639195
8,ROBBERY,46485,3393,3.348668
9,WEAPONS VIOLATION,43572,27989,3.138822


In [7]:
future_hotspots = pd.read_csv(
    DATA_DIR / "future_hotspots.csv"
)

area_risk = pd.read_csv(
    DATA_DIR / "latest_area_risk.csv"
)

crime_type_forecast = pd.read_csv(
    DATA_DIR / "api_crime_type_forecast.csv"
)

print(
    "Future hotspots:",
    future_hotspots.shape
)

print(
    "Area risk:",
    area_risk.shape
)

print(
    "Crime type forecast:",
    crime_type_forecast.shape
)

Future hotspots: (734, 23)
Area risk: (734, 24)
Crime type forecast: (10, 4)


In [8]:
future_hotspots_api = (
    future_hotspots[
        [
            'hotspot_rank',
            'week',
            'grid_id',
            'grid_lat',
            'grid_lon',
            'predicted_crimes',
            'risk_level'
        ]
    ]
    .copy()
)

future_hotspots_api.head()

,hotspot_rank,week,grid_id,grid_lat,grid_lon,predicted_crimes,risk_level
0,1,2026-10-05,41.89_-87.63,41.89,-87.63,60.836991,HIGH
1,2,2026-10-05,41.88_-87.63,41.88,-87.63,55.220352,HIGH
2,3,2026-10-05,41.9_-87.63,41.90,-87.63,32.411434,HIGH
3,4,2026-10-05,41.89_-87.62,41.89,-87.62,32.239775,HIGH
4,5,2026-10-05,41.76_-87.56,41.76,-87.56,22.550982,HIGH


In [9]:
future_hotspots_api[
    'predicted_crimes'
] = (
    future_hotspots_api[
        'predicted_crimes'
    ]
    .round(2)
)

In [10]:
hotspot_file = (
    DATA_DIR / "hotspots.csv"
)

if hotspot_file.exists():

    hotspots = pd.read_csv(
        hotspot_file
    )

    print(
        "Historical hotspots:",
        hotspots.shape
    )

else:

    hotspots = pd.DataFrame()

    print(
        "No hotspots.csv found"
    )

No hotspots.csv found


In [11]:
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(
    DB_PATH
)

print("Database created ✅")

Database created ✅


In [15]:
crime_summary.to_sql(
    "crime_summary",
    conn,
    if_exists="replace",
    index=False
)

monthly_crimes.to_sql(
    "monthly_crimes",
    conn,
    if_exists="replace",
    index=False
)

crime_types.to_sql(
    "crime_types",
    conn,
    if_exists="replace",
    index=False
)

future_hotspots_api.to_sql(
    "future_hotspots",
    conn,
    if_exists="replace",
    index=False
)

area_risk.to_sql(
    "area_risk",
    conn,
    if_exists="replace",
    index=False
)

crime_type_forecast.to_sql(
    "crime_type_forecast",
    conn,
    if_exists="replace",
    index=False
)

if not hotspots.empty:
    hotspots.to_sql(
        "hotspots",
        conn,
        if_exists="replace",
        index=False
    )

conn.commit()

print("Tables created ✅")

Tables created ✅


In [16]:
cursor = conn.cursor()

cursor.execute("""
CREATE INDEX IF NOT EXISTS
idx_future_hotspots_risk
ON future_hotspots(risk_level)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS
idx_future_hotspots_rank
ON future_hotspots(hotspot_rank)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS
idx_crime_types_count
ON crime_types(crime_count)
""")

conn.commit()

In [17]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
    """,
    conn
)

tables

,name
0,area_risk
1,crime_summary
2,crime_type_forecast
3,crime_types
4,future_hotspots
5,monthly_crimes


In [18]:
for table in tables['name']:

    result = pd.read_sql_query(
        f"""
        SELECT COUNT(*) AS rows
        FROM "{table}"
        """,
        conn
    )

    print(
        table,
        "→",
        result.iloc[0]['rows']
    )

area_risk → 734
crime_summary → 4
crime_type_forecast → 10
crime_types → 31
future_hotspots → 734
monthly_crimes → 70


In [19]:
pd.read_sql_query(
    """
    SELECT
        hotspot_rank,
        grid_lat,
        grid_lon,
        predicted_crimes,
        risk_level

    FROM future_hotspots

    ORDER BY hotspot_rank

    LIMIT 10
    """,
    conn
)

,hotspot_rank,grid_lat,grid_lon,predicted_crimes,risk_level
0,1,41.89,-87.63,60.84,HIGH
1,2,41.88,-87.63,55.22,HIGH
2,3,41.90,-87.63,32.41,HIGH
3,4,41.89,-87.62,32.24,HIGH
4,5,41.76,-87.56,22.55,HIGH
5,6,41.87,-87.63,22.41,HIGH
6,7,41.77,-87.58,21.96,HIGH
7,8,41.88,-87.73,21.65,HIGH
8,9,41.75,-87.61,20.87,HIGH
9,10,41.75,-87.60,20.33,HIGH


In [20]:
conn.close()

size_mb = (
    DB_PATH.stat().st_size
    / 1024
    / 1024
)

print(
    f"Database size: {size_mb:.2f} MB"
)

Database size: 0.21 MB
